# 第9回　効果量とp値批判
## ―― 「有意」であることと「重要」であることは、まったく別

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

前回（第8回）は「たくさん試すと偽の有意が出る」だった。今回は逆 ―― **本物の有意でも、それが重要とは限らない**。p値の最大の弱点と、その処方箋である **効果量** を学ぶ。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. 直感クイズ ―― ほんの少しの差を、巨大な標本で調べる

2つの教育法 A と B。本当の平均点の差は、たった **0.5点**（100点満点・SD=10に対してごくわずか）。実質的にはどうでもいい差だ。

この差を、標本サイズ $n$ をだんだん大きくして t 検定する。1つの標本だと p値は運でぶれるので、各 $n$ で **200回くりかえした p値の中央値**（典型値）を見る。

**問い：n を 100 → … → 10万 と増やすと、典型的な p値はどうなる？**　予想を決めてから ▶。

In [ ]:
rng = np.random.default_rng(9)
真の差, σ = 0.5, 10.0     # 本当の平均差はわずか0.5点（σ=10に対してごく小さい→効果量d=0.05）
nリスト = [100, 1000, 10000, 100000]

def 典型的なp値(n, reps=200):
    ps = [stats.ttest_ind(rng.normal(50, σ, n), rng.normal(50 + 真の差, σ, n)).pvalue
          for _ in range(reps)]
    return np.median(ps)

print("本当の差は 0.5点（ほぼ無意味）。それでも n を増やすと…")
pたち = []
for n in nリスト:
    p = 典型的なp値(n)
    pたち.append(p)
    print(f"  n={n:>6}： 典型的な p = {p:.4f}{'　★有意(p<0.05)' if p < 0.05 else ''}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(nリスト, pたち, "o-", color="#e8503a")
plt.axhline(0.05, ls="--", color="gray", label="有意水準 0.05")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("標本サイズ n（対数）"); plt.ylabel("典型的なp値（対数）")
plt.title("差は0.5点のまま。なのに n を増やすだけで p値は下がり続ける")
plt.legend(); plt.show()

**差は 0.5点のまま変わらないのに、$n$ を大きくするだけで p値はいくらでも小さくなる。** 10万人も調べれば、どうでもいい差が「p<0.001 で有意！」になる。

理由：t検定の統計量は（差の大きさ）×$\sqrt{n}$ におよそ比例する。差が小さくても、$n$ を増やせば統計量は大きくなり、p値は下がる。**p値は「差の重要さ」ではなく、半分は「標本の大きさ」を映しているだけ**だ。

---
## 2. 効果量 ―― 「差の大きさ」そのものを測る

p値が当てにならないなら、何を見るか。**効果量（effect size）** ―― 標本サイズに振り回されない、差の大きさそのものの指標だ。

2群の平均差には **Cohen's d** を使う：

$$d = \frac{\bar{x}_A - \bar{x}_B}{s}\quad(\text{差が、標準偏差いくつぶんか})$$

目安は $d=0.2$ 小、$0.5$ 中、$0.8$ 大。さっきの「0.5点差」を d で測ると？

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    s_pooled = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (b.mean() - a.mean()) / s_pooled

print("同じ『0.5点差』を、n を変えて効果量 d で測る：")
for n in [100, 100000]:
    A = rng.normal(50, σ, n)
    B = rng.normal(50 + 真の差, σ, n)
    p = stats.ttest_ind(A, B).pvalue
    d = cohens_d(A, B)
    print(f"  n={n:>6}： p={p:.4f}, 効果量 d={d:.3f}（理論 d=0.5/10={真の差/σ:.2f}＝ごく小）")
print("\n→ p値は n で激変するが、効果量 d は本当の差（≈0.05）の小ささを正しく映す。")

**p値は $n$ で激変するが、効果量はぶれない。** 0.5点差は d≈0.05 ―― 「小」の基準 0.2 にも遠く及ばない、無視してよい差だと分かる。

だから結果は **「有意かどうか」だけでなく、必ず「効果量はどれだけか」をセットで** 見る。できれば効果量に信頼区間を付けて「どれくらいの差が、どれくらいの確かさであるか」を語る。

---
## 3. 4つの組み合わせ ―― 有意と重要は独立

「統計的に有意か」と「実質的に重要（効果量が大きい）か」は、別々の軸だ。組み合わせは4通りある。

とくに危ないのが2つ：**「小さいが有意」**（大標本でどうでもいい差が有意に）と、**「大きいが非有意」**（小標本で本物の差を見逃す）。実例を作ろう。

In [ ]:
# ① 小さいが有意：巨大標本で、ごく小さな差
A1 = rng.normal(50, 10, 50000); B1 = rng.normal(50.5, 10, 50000)
p1, d1 = stats.ttest_ind(A1, B1).pvalue, cohens_d(A1, B1)
# ② 大きいが非有意：小標本で、はっきりした差
A2 = rng.normal(50, 10, 6); B2 = rng.normal(58, 10, 6)
p2, d2 = stats.ttest_ind(A2, B2).pvalue, cohens_d(A2, B2)

print("① 小さいが有意 ：", f"p={p1:.4f}（有意）だが 効果量 d={d1:.2f}（ごく小）→ 実質どうでもいい")
print("② 大きいが非有意：", f"p={p2:.3f}（非有意）だが 効果量 d={d2:.2f}（大）→ 見逃しかも（標本不足）")
print("\np値だけ見ると①を『発見』と誤り、②を『差なし』と切り捨ててしまう。")

**①をp値だけで「重要な発見」と誤り、②を「差はなかった」と切り捨てる** ―― これがp値偏重の典型的な誤用だ。

近年の **再現性危機**（多くの“有意な発見”が追試で再現しない問題）の一因も、ここにある。p<0.05 という関門だけを見て、効果量・事前登録（第8回）・サンプル設計を軽視してきたことのツケだ。

> 💬 **p値との正しい付き合い方**
> 
> p値は「偶然だけでこの差が出る出やすさ」を測るだけで、**差の大きさでも、仮説が正しい確率でも、結果の重要さでもない**。「有意差が出た」と聞いたら問う ―― **『効果量は？ それは実生活で意味のある大きさ？』**。p値と効果量、両方をそろえて初めて結論できる。


---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| p値の弱点 | 標本 $n$ に依存。$n$ を増やせば無意味な差でも有意になる |
| 効果量 | 差の大きさそのもの（Cohen's d など）。$n$ に振り回されない |
| 有意 ≠ 重要 | 「小さいが有意」「大きいが非有意」がありうる |
| 報告の作法 | p値だけでなく、効果量（＋信頼区間）をセットで示す |

- p値は半分「標本の大きさ」を映すだけ。**有意であることは、重要であることを意味しない。**
- 「有意差が出た」と聞いたら **「効果量は？」** を問う。

> **課題（Moodle）**：出力からp値と効果量を読み分け（自動採点）＋「この“統計的有意差”は実質的に重要か」の批判的記述。詳しくはMoodleの第9回課題を見ること。

> **次回予告**：第10回「因果推論入門：RCTと観察研究」。ここからは因果の話。「サプリを飲んだ人は健康だった。サプリのおかげ？ それとも、もともと健康志向の人が飲んだだけ？」